In [2]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 1.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 146.2 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.1 kB/s  0:00:41m0:00:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 475.7 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 2.1 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]


In [9]:
!pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.8 MB/s  0:00:00 eta 0:00:01


In [11]:
!pip install accelerate

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct"


In [3]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

In [12]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [14]:
print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant



In [16]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


In [17]:
model_inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  35127,    752,    264,
           2805,  16800,    311,   3460,   4128,   1614,     13, 151645,    198,
         151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [22]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.7,   # randomness, 0 = deterministic, >1 = more creative
    top_p=0.9,         # nucleus sampling, picks tokens from cumulative probability 0.9
    # top_k=50,          # picks from top 50 probable tokens
    do_sample=True     # must be True to use temperature, top_p, or top_k
)

In [ ]:
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=512,
#     do_sample=False    # greedy decoding, always picks most probable token
# )

In [23]:
generated_ids

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  35127,    752,    264,
           2805,  16800,    311,   3460,   4128,   1614,     13, 151645,    198,
         151644,  77091,    198,  39814,      0,    362,   3460,   4128,   1614,
            320,   4086,     44,      8,    374,    264,    943,    315,  20443,
          11229,   1614,   6188,    311,   1882,    323,   6923,   3738,  12681,
           1467,     13,   4220,   4119,    525,  11136,   3118,    389,   5538,
           6832,  12538,     11,   7945,  42578,  77235,     11,    892,   2138,
           1105,    311,   3535,   2266,     11,   6923,  55787,  14507,     11,
            323,   3705,    264,   6884,   2088,    315,   5810,   4128,   9079,
            382,   1592,  17452,    315,    444,  10994,     82,   2924,   1447,
             16,     13,   3

In [24]:
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

In [25]:
generated_ids

[tensor([ 39814,      0,    362,   3460,   4128,   1614,    320,   4086,     44,
              8,    374,    264,    943,    315,  20443,  11229,   1614,   6188,
            311,   1882,    323,   6923,   3738,  12681,   1467,     13,   4220,
           4119,    525,  11136,   3118,    389,   5538,   6832,  12538,     11,
           7945,  42578,  77235,     11,    892,   2138,   1105,    311,   3535,
           2266,     11,   6923,  55787,  14507,     11,    323,   3705,    264,
           6884,   2088,    315,   5810,   4128,   9079,    382,   1592,  17452,
            315,    444,  10994,     82,   2924,   1447,     16,     13,   3070,
           1695,  95518,    444,  10994,     82,    525,   5990,   1602,   3460,
             11,   8482,  11728,    476,   1496,  32051,    315,   5029,     13,
           1096,   1379,   6147,   1105,    311,  12322,   6351,  12624,    304,
           4128,    821,    382,     17,     13,   3070,  36930,   2885,  95518,
           2379,    525,  16

In [32]:
len([ 39814,      0,    362,   3460,   4128,   1614,    320,   4086,     44,
              8,    374,    264,    943,    315,  20443,  11229,   1614,   6188,
            311,   1882,    323,   6923,   3738,  12681,   1467,     13,   4220,
           4119,    525,  11136,   3118,    389,   5538,   6832,  12538,     11,
           7945,  42578,  77235,     11,    892,   2138,   1105,    311,   3535,
           2266,     11,   6923,  55787,  14507,     11,    323,   3705,    264,
           6884,   2088,    315,   5810,   4128,   9079,    382,   1592,  17452,
            315,    444,  10994,     82,   2924,   1447,     16,     13,   3070,
           1695,  95518,    444,  10994,     82,    525,   5990,   1602,   3460,
             11,   8482,  11728,    476,   1496,  32051,    315,   5029,     13,
           1096,   1379,   6147,   1105,    311,  12322,   6351,  12624,    304,
           4128,    821,    382,     17,     13,   3070,  36930,   2885,  95518,
           2379,    525,  16176,    389,  12767,  14713,    315,   1467,    821,
            504,    279,   7602,     11,   6467,     11,    323,   1008,   8173,
             11,  27362,   1105,    311,   3960,    264,   7205,   2088,    315,
          64667,  83789,    323,   6540,    382,     18,     13,   3070,  50359,
          95518,    444,  10994,     82,    646,    387,   1483,    369,   5257,
           5810,   4128,   8692,   9079,   1741,    438,   1467,   9755,     11,
          14468,     11,   3405,  35764,     11,  28285,   2022,     11,    323,
            803,    382,     19,     13,   3070,   3306,   7175,  95518,   3085,
          82687,   1075,   6236,  61905,    323,   7517,   1663,  13009,     11,
            444,  10994,     82,    646,  16579,    304,  20753,  21276,    448,
           3847,     11,   8241,  34549,    323,   2266,   1832,   9760,  14507,
            382,     20,     13,   3070,  65390,    938,  21144,    804,  95518,
           5976,   7988,     11,    444,  10994,     82,   1083,   4828,  30208,
          10520,   5435,    311,  15470,     11,  12345,     11,    323,    279,
           4650,  61751,    315,   7907,   2213,    382,  40381,    315,   1632,
          21309,   3460,   4128,   4119,   2924,   5085,    594,    425,   3399,
             11,   5264,  15469,    594,    479,   2828,   4013,     11,    323,
          54364,  14817,    594,   1207,  16948,     13,   4220,   4119,   3060,
            311,  37580,     11,  17461,    279,  22711,    315,   1128,  15235,
            646,  11075,    304,   5810,   4128,   8660,    323,   9471,     13,
         151645])

289

In [30]:
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [31]:
response

"Sure! A large language model (LLM) is a type of artificial intelligence model designed to process and generate human-like text. These models are typically based on deep learning techniques, particularly transformer architectures, which allow them to understand context, generate coherent responses, and handle a wide range of natural language tasks.\n\nKey characteristics of LLMs include:\n\n1. **Size**: LLMs are usually very large, containing millions or even billions of parameters. This size allows them to capture complex patterns in language data.\n\n2. **Training Data**: They are trained on vast amounts of text data from the internet, books, and other sources, enabling them to learn a broad range of linguistic nuances and knowledge.\n\n3. **Applications**: LLMs can be used for various natural language processing tasks such as text completion, translation, question answering, summarization, and more.\n\n4. **Interactivity**: With advancements like chatbots and conversational agents, 

In [33]:
!nvidia-smi

Fri Oct  3 19:00:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.09              Driver Version: 580.82.09      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
| 30%   25C    P8             17W /  350W |   14918MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
